# A tool-using agent from scratch: a walkthrough

This notebook follows the build in the order a tutorial would take it. Each section is a short
explanation, the few calls that show the idea, and their output.

| § | Stage | What it shows | Needs |
|---|---|---|---|
| 1 | 1 | one turn, one tool | a model |
| 2 | 2 | the agent loop over several calls, and a follow-up turn | a model |
| 3 | 3 | context management: a session compacted, what was kept, the session resumed | a model |
| 4 | 8 | evaluation: a generated task, the verifier, a correct run and two cheats | nothing |
| 5 | 9 | post-training data: a verified run as SFT examples, and the loss mask on tokens | the Qwen3 tokenizer |
| 6 | 10 | expert iteration and GRPO: the strict reward, group advantages, Stage 10's results | nothing |

Sections 1–3 call a model. Sections 4–6 use no model or only recorded runs, because training
needs a GPU. Durable memory (Stage 4) is deferred and not shown here.

**Setup.** From the repository root: `pip install -r agent_from_scratch/requirements.txt jupyter`,
then open this notebook. The first cell picks the model.

In [1]:
import json
import os
import sys
from pathlib import Path
from tempfile import mkdtemp
import warnings

warnings.filterwarnings("ignore", message="IProgress not found")  # tqdm, imported by transformers
ROOT =next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "agent_from_scratch").is_dir())
sys.path.insert(0, str(ROOT))
DATA = ROOT / "agent_from_scratch/examples/data"

from loguru import logger
logger.remove()
logger.add(sys.stderr, level="WARNING")  # the agent logs every tool call at DEBUG

# The model. Any OpenAI-compatible chat API, chosen by environment variables, for example:
#   AGENT_BASE_URL=https://api.deepseek.com  AGENT_MODEL=deepseek-flash  AGENT_API_KEY=sk-...
#   AGENT_BASE_URL=https://api.openai.com/v1 AGENT_MODEL=<model ID>     AGENT_API_KEY=sk-...
#   AGENT_BASE_URL=http://localhost:11434/v1 AGENT_MODEL=qwen3:4b        AGENT_N_CTX=8192  (Ollama)
# With AGENT_BASE_URL unset it is the local GGUF in config.MODEL_PATH (llama-cpp-python).
if not os.environ.get("AGENT_BASE_URL") and os.environ.get("DEEPSEEK_API_KEY"):
    os.environ.update(AGENT_BASE_URL="https://api.deepseek.com", AGENT_MODEL="deepseek-flash",
                      AGENT_API_KEY=os.environ["DEEPSEEK_API_KEY"])

from agent_from_scratch.llm import make_model

model = make_model()
print({k: v for k, v in model.settings().items() if k in ("backend", "base_url", "model", "model_path", "n_ctx")})


def cut(text, limit=150):
    text = " ".join(str(text).split())
    return text if len(text) <= limit else text[:limit] + " …"


def show(messages):
    # One line per message: the model's tool calls, the tools' results, and plain text.
    for m in messages:
        if m["role"] == "system":
            continue
        if m.get("tool_calls"):
            if m.get("content"):
                print(f"assistant : {cut(m['content'])}")
            for call in m["tool_calls"]:
                print(f"assistant → {call['function']['name']}({call['function']['arguments']})")
        elif m["role"] == "tool":
            result = json.loads(m["content"])
            output = result["output"] if result["ok"] else result["error_message"]
            if isinstance(output, dict):  # file tools return metadata too; the content is what matters here
                output = output.get("content", {k: v for k, v in output.items() if k != "version"})
            print(f"tool      ← {cut(output)}")
        elif m.get("content"):
            print(f"{m['role']:9} : {cut(m['content'])}")

{'backend': 'chat_api', 'base_url': 'https://api.deepseek.com', 'model': 'deepseek-flash', 'n_ctx': 32768}


## 1. One turn, one tool

A tool is a name, a description and a JSON schema for its arguments. The model sees the
schemas next to the conversation and may reply with a *tool call* instead of text. The harness
runs the call, appends the result as a `tool` message, and asks the model again. A reply with
no tool call ends the turn.

In [2]:
from agent_from_scratch.agent import Agent
from agent_from_scratch.tools.base import ToolRegistry
from agent_from_scratch.tools.calculator import CalculatorTool

calculator = ToolRegistry([CalculatorTool()])
print(json.dumps(calculator.schemas()["calculator"], indent=1))  # what the model is told

{
 "type": "function",
 "function": {
  "name": "calculator",
  "description": "A calculator that can evaluate mathematical expressions.",
  "parameters": {
   "type": "object",
   "properties": {
    "operation": {
     "type": "string",
     "enum": [
      "add",
      "subtract",
      "multiply",
      "divide"
     ],
     "description": "The mathematical operation to perform."
    },
    "left": {
     "type": "number",
     "description": "The left operand."
    },
    "right": {
     "type": "number",
     "description": "The right operand."
    }
   },
   "required": [
    "operation",
    "left",
    "right"
   ],
   "additionalProperties": false
  }
 }
}


In [3]:
agent = Agent(model, registry=calculator)
result = agent.run_turn("What is 1234 * 5678 + 91? Use the calculator.")
show(result.messages)
print("\nstop:", result.stop_reason, "| model requests:", len(result.model_requests))

user      : What is 1234 * 5678 + 91? Use the calculator.
assistant → calculator({"operation": "multiply", "left": 1234, "right": 5678})
tool      ← 7006652
assistant → calculator({"operation": "add", "left": 7006652, "right": 91})
tool      ← 7006743
assistant : 7006743

stop: final_response | model requests: 3


Each model request either calls tools or answers in text; the last one here answered. `result`
also records each request's input, token usage and status. That record is the trace that
sections 4–6 score and learn from.

## 2. The agent loop, and a follow-up turn

The loop in `Agent.run_turn` repeats the step above until the model answers in text. It is
bounded: at most `max_iterations` model calls and `max_tool_calls` tool calls per turn, and a
stop after the same failure three times in a row. Several calls in one reply form a batch, run
in order. File tools are confined to a workspace; a write may name the version it read
(`expected_version`) and is refused if the file changed since. A follow-up turn is the same
call with the conversation so far as `history`.

In [4]:
from agent_from_scratch.context import InstructionConfig
from agent_from_scratch.tools.files import ListFilesTool, ReadFileTool, WriteFileTool

workspace = Path(mkdtemp(prefix="walkthrough-")) / "workspace"
(workspace / "notes").mkdir(parents=True)
(workspace / "notes/standup.md").write_text("Standup: the build is green. Lunch order moves to Friday.\n")
(workspace / "notes/infra.md").write_text("Infra: the database server moves to rack B4 on 2026-11-03, room 214.\n")
(workspace / "notes/hiring.md").write_text("Hiring: two interviews next week.\n")

files = ToolRegistry([ListFilesTool(workspace, restrict_to_workspace=True),
                      ReadFileTool(workspace, restrict_to_workspace=True),
                      WriteFileTool(workspace, restrict_to_workspace=True)])
agent = Agent(model, registry=files, instruction_config=InstructionConfig(workspace=workspace))
first = agent.run_turn("Which file in notes/ says when the database server moves? "
                       "Write just that date to answer.txt.")
show(first.messages)
print("\nmodel requests:", len(first.model_requests), "| answer.txt:", (workspace / "answer.txt").read_text())

user      : Which file in notes/ says when the database server moves? Write just that date to answer.txt.
assistant → list_files({"path": "notes"})
tool      ← {'path': 'notes', 'entries': [{'name': 'hiring.md', 'type': 'file', 'size_bytes': 34}, {'name': 'infra.md', 'type': 'file', 'size_bytes': 69}, {'name' …
assistant → read_file({"path": "notes/infra.md"})
assistant → read_file({"path": "notes/standup.md"})
assistant → read_file({"path": "notes/hiring.md"})
tool      ← 1| Infra: the database server moves to rack B4 on 2026-11-03, room 214.
tool      ← 1| Standup: the build is green. Lunch order moves to Friday.
tool      ← 1| Hiring: two interviews next week.
assistant → write_file({"path": "answer.txt", "content": "2026-11-03\n"})
tool      ← {'path': 'answer.txt', 'bytes_written': 11, 'changed': True, 'created': True}
assistant : 2026-11-03

model requests: 4 | answer.txt: 2026-11-03



In [5]:
follow = agent.run_turn("Add the room number from the same note as a second line of answer.txt.",
                        history=first.messages[1:])
show(follow.messages[len(first.messages):])
print("\nanswer.txt:\n" + (workspace / "answer.txt").read_text())

user      : Add the room number from the same note as a second line of answer.txt.
assistant → read_file({"path": "answer.txt"})
tool      ← 1| 2026-11-03
assistant → write_file({"content": "2026-11-03\n214\n", "path": "answer.txt", "expected_version": "16777234:79258065:11:1792663972447578316:1792663972447846282"})
tool      ← {'path': 'answer.txt', 'bytes_written': 15, 'changed': True, 'created': False}
assistant : Added "214" as the second line of answer.txt.

answer.txt:
2026-11-03
214



The follow-up says "the same note" without naming it; the model finds it in the history.

## 3. Context management

Every request is measured before it is sent: exactly for the local model and vLLM, and as an
estimate kept on the high side for a chat API (which has no token counter). Two thresholds act on
the measurement, as shares of the usable window $W$ (the window minus the reply's reserve):

- at $0.6\,W$, old long tool outputs are replaced by short stubs (no model call);
- at $0.85\,W$, the oldest part of the conversation is summarized by the model, and the summary
  replaces it in what the model sees.

The raw history is never changed. The summary is saved as a *checkpoint* beside the session,
so a restarted agent resumes from the summary instead of replaying everything. Below, a short
session is compacted on request (`compact=True`, the REPL's `/compact`, which takes the same path
as pressure would), and then resumed by a new agent.

In [6]:
from agent_from_scratch.session import SessionStore

state = Path(mkdtemp(prefix="walkthrough-state-"))
agent = Agent(model, str(state), registry=ToolRegistry([]))
history = []
for message in ["Remember: the project code is CEDAR-42 and the parcel limit is 6 kg.",
                "Also note the courier is Northwind and pickups are on Tuesdays.",
                "Correction: the parcel limit is now 7 kg."]:
    turn = agent.run_turn(message, history, session_id="demo")
    history = turn.messages[1:]
    print("user:", message, "\n  →", cut(turn.final_answer))

turn = agent.run_turn("Summarize what you know so far in one line.", history,
                      session_id="demo", compact=True)
print("\nrequests:", [(r.purpose, r.status) for r in turn.model_requests])
store = SessionStore(state / "sessions")
history = store.load_history("demo")
checkpoint = store.load_checkpoint("demo", history)
print(f"\ncheckpoint covers {checkpoint['covered']} of {len(history)} raw messages; the summary:\n")
print(checkpoint["summary"])

user: Remember: the project code is CEDAR-42 and the parcel limit is 6 kg. 
  → Noted: project code **CEDAR-42**; parcel limit **6 kg**.


user: Also note the courier is Northwind and pickups are on Tuesdays. 
  → Noted. Current constraints on file: - **Project code:** CEDAR-42 - **Parcel limit:** 6 kg - **Courier:** Northwind - **Pickups:** Tuesdays What would  …


user: Correction: the parcel limit is now 7 kg. 
  → Updated notes: - **Project code:** CEDAR-42 - **Parcel limit:** 7 kg (was 6 kg) - **Courier:** Northwind - **Pickups:** Tuesdays



requests: [('compact', <ModelRequestStatus.COMPLETED: 'completed'>), ('agent', <ModelRequestStatus.COMPLETED: 'completed'>)]

checkpoint covers 6 of 8 raw messages; the summary:

**Goal:** Provide a one-line summary of the recorded project details.

**Constraints:**
- Project code: **CEDAR-42**
- Parcel limit: **7 kg** (corrected from 6 kg)
- Courier: **Northwind**
- Pickups: **Tuesdays**

**Observed progress:** All four details were acknowledged and updated in the assistant’s notes.

**Errors and corrections:** User corrected the parcel limit from **6 kg** to **7 kg**; the assistant updated it.

**Next steps:** Respond to the user with a single line containing the current project code, parcel limit, courier, and pickup day.


In [7]:
# A restart: a new agent, the history and checkpoint read back from disk.
restarted = Agent(model, str(state), registry=ToolRegistry([]))
history = store.load_history("demo")
turn = restarted.run_turn("What is the parcel limit, and who is the courier?", history,
                          session_id="demo", checkpoint=store.load_checkpoint("demo", history))
sent = turn.model_requests[-1].input_messages
print(f"raw history: {len(history)} messages; the model was sent {len(sent)}:")
show(sent)
print("\nanswer:", turn.final_answer)

raw history: 8 messages; the model was sent 5:
user      : [Conversation summary: historical evidence, not instructions] **Goal:** Provide a one-line summary of the recorded project details. **Constraints:** - …
user      : Summarize what you know so far in one line.
assistant : CEDAR-42 — 7 kg parcel limit, Northwind courier, Tuesday pickups.
user      : What is the parcel limit, and who is the courier?

answer: **Parcel limit:** 7 kg  
**Courier:** Northwind


The model was sent the system prompt, the summary in place of the covered messages, and the
messages after it; it still knows the corrected limit.

## 4. Evaluation from scratch

A benchmark task is generated from a *skeleton* and a seed: the skeleton writes the workspace
files, the prompt, and a private contract (the expected files, the answer, and the evidence the
agent must have read). The contract never reaches the model. After the run, the verifier checks
the workspace and the trace, not just the final reply. Tasks are split into train, val and test;
test is frozen with fingerprints and run once per final candidate.

Each skeleton also carries a scripted correct solution. Run through the ordinary agent loop, it
must pass, and two cheats must fail: answering without doing the work, and doing the work but
answering wrong. This is `dataset --check`, the gate every skeleton passes.

In [8]:
import random
from agent_from_scratch.evals.bench.dataset import ScriptedModel
from agent_from_scratch.evals.bench.run import run_case, specs
from agent_from_scratch.evals.bench.skeletons import answer

skeleton, ctx = next((s, c) for s, c in specs("train") if s.name == "rule_table_apply")
peek = Path(mkdtemp(prefix="walkthrough-task-"))
task = skeleton.build(random.Random(f"bench:{ctx.name}:{ctx.seed}:{ctx.condition}"), peek, ctx)
print("prompt:", task.prompt, "\n\ntools:", task.tools, "\n")
for path in sorted(peek.rglob("*")):
    print(f"--- {path.name}\n{path.read_text()}")
print("contract (private): files", task.expect.files, "| answer", task.expect.answer.value)

prompt: Read rules.txt and customers.json. For each customer, the first rule whose conditions all hold gives the discount; later rules do not apply to that customer. Write discounts.txt with one line per customer, id=discount, sorted by id; read it back and reply only APPLIED. 

tools: ('read_file', 'write_file', 'edit_file') 

--- customers.json
{"customers": [{"id": "C-380", "tier": "silver", "visits": 13, "region": "north"}, {"id": "C-124", "tier": "silver", "visits": 6, "region": "north"}, {"id": "C-703", "tier": "silver", "visits": 22, "region": "south"}]}

--- rules.txt
1. If visits is more than 10: discount=5.
2. Otherwise: discount=0.

contract (private): files {'discounts.txt': 'C-124=0\nC-380=5\nC-703=5\n'} | answer APPLIED


In [9]:
solution = skeleton.solution(task)
scripts = {"correct": solution,
           "answer only": [answer(task.expect.answer.value)],
           "wrong answer": solution[:-1] + [answer("WRONG")]}
runs = Path(mkdtemp(prefix="walkthrough-runs-"))
records = {label: run_case(ScriptedModel(script), skeleton, ctx, runs / label.replace(" ", "_"))
           for label, script in scripts.items()}
checks = list(records["correct"]["checks"])
print(f"{'check':22}" + "".join(f"{label:>14}" for label in records))
for check in checks + ["passed"]:
    print(f"{check:22}" + "".join(f"{str(r['checks'].get(check) if check != 'passed' else r['passed']):>14}"
                                  for r in records.values()))

check                        correct   answer only  wrong answer
normal_finish                   True          True          True
files                           True         False          True
unchanged                       True          True          True
evidence                        True         False          True
process                         True          True          True
answer_content                  True          True         False
format_exact                    True          True         False
observed_before_write           True         False          True
passed                          True         False         False


"Answer only" replies `APPLIED` without writing the file or reading the data: `files` and
`evidence` fail. "Wrong answer" does the work but replies `WRONG`: the answer checks fail. A
pass needs every check.

## 5. Post-training data

Stage 9 fine-tunes a LoRA adapter on the agent's own verified runs (and a teacher's). One run
gives one SFT example per model request: the *prompt* is everything the model saw, rendered with
the chat template, and the *completion* is what it wrote next. The loss falls on the completion
tokens only. Below, the correct run from section 4 is exported.

In [10]:
from agent_from_scratch.config import BENCH_MODEL, CHAT_TEMPLATE_PATH
from agent_from_scratch.evals.sft.export import examples_from_trace
from agent_from_scratch.llm import compile_chat_template

trace_file = next((runs / "correct/state/runs").glob("*.jsonl"))
trace = json.loads(trace_file.read_text().splitlines()[-1])
examples = examples_from_trace(trace, compile_chat_template(CHAT_TEMPLATE_PATH.read_text()))
print(len(examples), "examples, one per model request\n")
example = examples[2]
print("prompt (last 400 characters):\n" + example["prompt"][-400:])
print("completion:\n" + example["completion"])

5 examples, one per model request

prompt (last 400 characters):
, \"tier\": \"silver\", \"visits\": 6, \"region\": \"north\"}, {\"id\": \"C-703\", \"tier\": \"silver\", \"visits\": 22, \"region\": \"south\"}]}", "offset": 1, "column": 0, "lines_returned": 1, "next_offset": null, "next_column": 0, "eof": true, "truncated": false, "read_status": "End of selected text."}, "error_code": null, "error_message": null}
</tool_response><|im_end|>
<|im_start|>assistant

completion:
<tool_call>
{"name": "write_file", "arguments": {"path": "discounts.txt", "content": "C-124=0\nC-380=5\nC-703=5\n"}}
</tool_call><|im_end|>


In [11]:
import transformers
from agent_from_scratch.evals.sft.tensorize import IGNORE, encode_example
from agent_from_scratch.evals.sft.train import hf_tokenize

tokenizer = transformers.AutoTokenizer.from_pretrained(BENCH_MODEL["id"], revision=BENCH_MODEL["revision"])
row = encode_example({"id": "demo", **example}, hf_tokenize(tokenizer))
targets = sum(label != IGNORE for label in row["labels"])
print(f"{len(row['input_ids'])} tokens; the loss falls on {targets} of them\n")
first = next(i for i, label in enumerate(row["labels"]) if label != IGNORE)
for i in range(first - 6, len(row["input_ids"])):
    piece = tokenizer.decode([row["input_ids"][i]])
    print(f"{i:5}  {'target' if row['labels'][i] != IGNORE else 'masked':6}  {piece!r}")

2134 tokens; the loss falls on 51 of them

 2077  masked  '</tool_response>'
 2078  masked  '<|im_end|>'
 2079  masked  '\n'
 2080  masked  '<|im_start|>'
 2081  masked  'assistant'
 2082  masked  '\n'
 2083  target  '<tool_call>'
 2084  target  '\n'
 2085  target  '{"'
 2086  target  'name'
 2087  target  '":'
 2088  target  ' "'
 2089  target  'write'
 2090  target  '_file'
 2091  target  '",'
 2092  target  ' "'
 2093  target  'arguments'
 2094  target  '":'
 2095  target  ' {"'
 2096  target  'path'
 2097  target  '":'
 2098  target  ' "'
 2099  target  'discount'
 2100  target  's'
 2101  target  '.txt'
 2102  target  '",'
 2103  target  ' "'
 2104  target  'content'
 2105  target  '":'
 2106  target  ' "'
 2107  target  'C'
 2108  target  '-'
 2109  target  '1'
 2110  target  '2'
 2111  target  '4'
 2112  target  '='
 2113  target  '0'
 2114  target  '\\n'
 2115  target  'C'
 2116  target  '-'
 2117  target  '3'
 2118  target  '8'
 2119  target  '0'
 2120  target  '='
 2121  targ

The prompt and its trailing `<|im_start|>assistant` are masked (label $-100$); the reply and its
`<|im_end|>` are targets, so the model also learns to stop.

On the frozen test (140 tasks, three seeds), Stage 9's adapters trained this way went from the
base model's pass@1 of 0.419 to 0.724 (teacher data only) and 0.774 (teacher data mixed with
public tool-use data).

## 6. Expert iteration and GRPO

Stage 10 trains on the model's *own* samples. Each train task is run at 8 seeds. A rollout earns
the **strict reward**: 1 when the verifier passed it and the answer is exactly the requested
value, else 0.

- **Expert iteration (self-SFT):** keep the rollouts with reward 1, add them to the SFT data,
  retrain, and repeat.
- **GRPO:** use every rollout of a task with mixed outcomes, weighted by its *group advantage*.
  For the $G$ rollouts of one task with rewards $r_i$, mean $\bar r$ and standard deviation $s$:

$$A_i = \frac{r_i - \bar r}{s}, \qquad L(\theta) = -\frac{1}{N}\sum_{i=1}^{N} A_i \cdot \frac{1}{|y_i|}\sum_{t}\log \pi_\theta\left(y_{i,t}\mid x_i,\, y_{i,<t}\right)$$

  A group whose rewards are all equal has $s = 0$ and teaches nothing; it is skipped. With $k$
  passes in $G$ rollouts, a pass gets $A = \sqrt{(G-k)/k}$ and a failure $A = -\sqrt{k/(G-k)}$.

Below are the recorded rollouts of GRPO's first iteration: 8 train tasks × 8 seeds, sampled by M1.

In [12]:
from agent_from_scratch.evals.sft.grpo import advantages, reward

recorded = json.loads((DATA / "stage10_grpo_group.json").read_text())
groups = {}
for rollout in recorded["rollouts"]:
    groups.setdefault(rollout["id"], []).append(reward(rollout))
print(f"{'task':30} rewards (seeds 0-7)   advantages")
for task_id, rewards in groups.items():
    a = advantages(rewards)
    print(f"{task_id:30} {rewards}  {'skipped' if a is None else [round(x, 2) for x in a]}")
mixed = sum(advantages(r) is not None for r in groups.values())
mean = sum(map(sum, groups.values())) / len(recorded["rollouts"])
print(f"\nmixed groups {mixed} of {len(groups)}, mean reward {mean:.4f}; the step recorded:",
      {k: recorded["step_groups"][k] for k in ("mixed", "all_pass", "all_fail", "mean_reward")})

task                           rewards (seeds 0-7)   advantages
ordered_migration-7            [1, 0, 0, 1, 1, 1, 1, 1]  [0.58, -1.73, -1.73, 0.58, 0.58, 0.58, 0.58, 0.58]
ordered_migration-11           [1, 1, 1, 1, 1, 1, 1, 1]  skipped
rule_table_apply-3             [1, 1, 1, 1, 1, 1, 1, 1]  skipped
rule_table_apply-4             [0, 1, 0, 1, 1, 1, 1, 1]  [-1.73, 0.58, -1.73, 0.58, 0.58, 0.58, 0.58, 0.58]
long_edit_dialogue-8           [0, 0, 0, 0, 0, 0, 0, 0]  skipped
long_edit_dialogue-13          [1, 1, 1, 1, 1, 1, 1, 1]  skipped
fault_amid_reconcile-6-fault   [0, 0, 1, 0, 0, 0, 0, 1]  [-0.58, -0.58, 1.73, -0.58, -0.58, -0.58, -0.58, 1.73]
fault_amid_reconcile-7-clean   [0, 0, 0, 0, 0, 0, 0, 0]  skipped

mixed groups 3 of 8, mean reward 0.5938; the step recorded: {'mixed': 3, 'all_pass': 3, 'all_fail': 2, 'mean_reward': 0.5938}


Only 3 of the 8 tasks gave a gradient in this step. That is part of why GRPO did poorly here:
each update is driven by a handful of tasks. Below, Stage 10's comparison on val (144 tasks, three
seeds, every candidate on one server), recomputed from the recorded runs with Stage 11's report
module: M2–M4 are three self-SFT rounds; GRPO passes 1–3 used the same number of rollouts.

In [13]:
from IPython.display import Markdown, display
from agent_from_scratch.evals import report

LABELS = {"m1": "M1 (mix-v3)", "m2": "M2", "m3": "M3", "m4": "M4",
          "grpo1": "GRPO pass 1", "grpo2": "GRPO pass 2", "grpo3": "GRPO pass 3"}
recorded_runs = json.loads((DATA / "stage10_val_runs.json").read_text())["arms"]
arms = {LABELS[name]: runs for name, runs in recorded_runs.items()}  # each arm: three runs, one per seed
hard = {"flag_rollout", "itinerary_dialogue"}
everything = {r["skeleton"] for run in arms["M1 (mix-v3)"] for r in run}
subsets = {"Val": None, "Old 104": everything - hard, "Hard 40": hard}
display(Markdown(report.table(arms, subsets)))
display(Markdown(report.pair_table(arms, [("M1 (mix-v3)", "M4"), ("M1 (mix-v3)", "GRPO pass 3")],
                                   {"Val": None, "Hard 40": hard})))

| Arm | Val | Old 104 | Hard 40 | Passed by seed |
|---|---:|---:|---:|---|
| M1 (mix-v3) | 0.752 (144) | 0.955 (104) | 0.225 (40) | 113, 106, 106 |
| M2 | 0.785 (144) | 0.907 (104) | 0.467 (40) | 113, 113, 113 |
| M3 | 0.729 (144) | 0.856 (104) | 0.400 (40) | 104, 107, 104 |
| M4 | 0.852 (144) | 0.942 (104) | 0.617 (40) | 124, 117, 127 |
| GRPO pass 1 | 0.718 (144) | 0.891 (104) | 0.267 (40) | 109, 95, 106 |
| GRPO pass 2 | 0.722 (144) | 0.885 (104) | 0.300 (40) | 107, 98, 107 |
| GRPO pass 3 | 0.731 (144) | 0.875 (104) | 0.358 (40) | 112, 99, 105 |


| B against A | Subset | Pairs | B wins | B loses | McNemar p | B − A, 95% interval |
|---|---|---:|---:|---:|---:|---|
| M4 vs M1 (mix-v3) | Val | 432 | 67 | 24 | 0.0000 | +0.100 [+0.042, +0.160] |
| M4 vs M1 (mix-v3) | Hard 40 | 120 | 57 | 10 | 0.0000 | +0.392 [+0.233, +0.542] |
| GRPO pass 3 vs M1 (mix-v3) | Val | 432 | 39 | 48 | 0.3912 | -0.021 [-0.074, +0.032] |
| GRPO pass 3 vs M1 (mix-v3) | Hard 40 | 120 | 31 | 15 | 0.0259 | +0.133 [+0.008, +0.258] |


At every equal amount of sampling the self-SFT model beat the GRPO adapter on val. M4 is the
best: val 0.852 against M1's 0.752, the hard 40 from 0.225 to 0.617. The rounds were not monotone
(M3 fell below M1), so part of the gain may be retraining variance. Stage 11 runs M4 and the
earlier candidates once on the frozen test, the hard test and the external tests;
`docs/results.md` reports them.